# Day 1 - PySpark Basics

This notebook covers the fundamentals of PySpark including:
- Creating SparkSession & SparkContext
- RDD operations (parallelize, map, collect)
- Reading CSV files
- DataFrame operations
- Writing to Parquet format

## 1. Import Libraries

Suppress warnings for cleaner output.

In [1]:
import warnings
warnings.filterwarnings('ignore')

## 2. Create SparkSession

`SparkSession` is the unified entry point for Spark (introduced in Spark 2.0).
- `appName` - Name of the application
- `getOrCreate()` - Returns existing session or creates new one

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName('first session')\
    .getOrCreate()
sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/02 18:19:04 WARN Utils: Your hostname, biswajits, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/02 18:19:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/02 18:19:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Verify SparkSession

Display the SparkSession details including version and master URL.

In [3]:
spark

## 4. View SparkContext

`SparkContext` is the older entry point for RDD operations.

In [4]:
sc

<SparkContext master=local[*] appName=first session>

## 5. RDD Operations

### Create RDD from Python list
`parallelize()` converts a Python collection to a distributed RDD.

In [5]:
lst = [1, 2, 3, 4, 5]
new_1 = sc.parallelize(lst)

### Apply map transformation
`map()` applies a function to each element. This is a lazy transformation.

In [6]:
lst_2 = new_1.map(lambda x: x*10)

### Print RDD object
Note: This shows the RDD object, not the actual data.

In [7]:
print(lst_2)

PythonRDD[1] at RDD at PythonRDD.scala:59


### Collect results
`collect()` triggers computation and brings all data to the driver node.

In [8]:
result = lst_2.collect()

### Display result

In [9]:
print(result)

[10, 20, 30, 40, 50]


## 6. Reading CSV Files

### Method 1: Using spark.read.csv()
- `header=True` - First row contains column names
- `inferSchema=True` - Auto-detect data types

In [10]:
file_path = r"../dataset/Customers.csv"
dataset = spark.read.csv(file_path, header=True, inferSchema=True)

### Method 2: Without inferSchema
All columns are read as strings when `inferSchema=False` (default).

In [11]:
dataset = spark.read.csv(file_path, header=True)

### Method 3: Using format() API
More explicit way to specify options.

In [12]:
df = spark.read.format('csv')\
    .option('header', 'True')\
    .option('inferSchema', 'True')\
    .load(file_path)

## 7. DataFrame Operations

### View first 5 rows with take()

In [13]:
dataset.take(5)

[Row(CustomerID='1', FirstName='Jossef', LastName='Goldberg', Country='Germany', Score='350'),
 Row(CustomerID='2', FirstName='Kevin', LastName='Brown', Country='USA', Score='900'),
 Row(CustomerID='3', FirstName='Mary', LastName=None, Country='USA', Score='750'),
 Row(CustomerID='4', FirstName='Mark', LastName='Schwarz', Country='Germany', Score='500'),
 Row(CustomerID='5', FirstName='Anna', LastName='Adams', Country='USA', Score=None)]

### View data with the format() loaded DataFrame

In [14]:
df.take(5)

[Row(CustomerID=1, FirstName='Jossef', LastName='Goldberg', Country='Germany', Score=350),
 Row(CustomerID=2, FirstName='Kevin', LastName='Brown', Country='USA', Score=900),
 Row(CustomerID=3, FirstName='Mary', LastName=None, Country='USA', Score=750),
 Row(CustomerID=4, FirstName='Mark', LastName='Schwarz', Country='Germany', Score=500),
 Row(CustomerID=5, FirstName='Anna', LastName='Adams', Country='USA', Score=None)]

### Display schema
Shows column names and their data types.

In [15]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Score: integer (nullable = true)



### Pretty print the DataFrame
`show()` displays data in a tabular format.

In [16]:
df.show(5)

+----------+---------+--------+-------+-----+
|CustomerID|FirstName|LastName|Country|Score|
+----------+---------+--------+-------+-----+
|         1|   Jossef|Goldberg|Germany|  350|
|         2|    Kevin|   Brown|    USA|  900|
|         3|     Mary|    NULL|    USA|  750|
|         4|     Mark| Schwarz|Germany|  500|
|         5|     Anna|   Adams|    USA| NULL|
+----------+---------+--------+-------+-----+



## 8. Rename Column
`withColumnRenamed()` creates a new DataFrame with the renamed column.

In [17]:
df_1 = df.withColumnRenamed('CustomerID', 'ID')

### Verify the rename

In [18]:
df_1.show(5)

+---+---------+--------+-------+-----+
| ID|FirstName|LastName|Country|Score|
+---+---------+--------+-------+-----+
|  1|   Jossef|Goldberg|Germany|  350|
|  2|    Kevin|   Brown|    USA|  900|
|  3|     Mary|    NULL|    USA|  750|
|  4|     Mark| Schwarz|Germany|  500|
|  5|     Anna|   Adams|    USA| NULL|
+---+---------+--------+-------+-----+



## 9. Write to Parquet

Parquet is a columnar storage format optimized for analytics:
- Efficient compression
- Schema preservation
- Predicate pushdown

In [19]:
df_1.write.mode('overwrite').parquet('../dataset/parquet_output')

## 10. Stop Spark Session

Always stop the session to release cluster resources.

In [20]:
spark.stop()